In [0]:
%sql
SELECT 
customer_id,
ship_to_address,
state,
city,
region,
district,
clicked_items,
lon,
lat,
valid_from,
valid_to,
insert_date

FROM end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc limit 10

In [0]:
df = spark.sql("""
SELECT customer_id, clicked_items, lat, lon 
FROM end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc 
WHERE clicked_items IS NOT NULL
""")

from pyspark.sql.functions import explode, split, col

# Split clicked_items by ',' and explode
df_split = df.withColumn("item_value", explode(split(col("clicked_items"), ",")))

# Split item_value by ':' into item and value columns
df_final = df_split.withColumn("item", split(col("item_value"), ":")[0]) \
                   .withColumn("value", split(col("item_value"), ":")[1]) \
                   .select("customer_id", "lat", "lon", "item", "value")

In [0]:
from pyspark.sql.functions import regexp_replace

# Add row index
df_split_indexed = df_split.withColumn("row_idx", monotonically_increasing_id())

# Even rows: keep only lat, lon from even table
df_even = df_split_indexed.filter((col("row_idx") % 2) == 0) \
    .select("customer_id", "item_value") \
    .withColumnRenamed("item_value", "item_value_even") \
    .withColumn("item_value_even", regexp_replace(col("item_value_even"), r'[\["]', ''))

# Odd rows
df_odd = df_split_indexed.filter((col("row_idx") % 2) == 1) \
    .withColumnRenamed("item_value", "item_value_odd") \
    .withColumnRenamed("row_idx", "row_idx_odd") \
    .withColumn("item_value_odd", regexp_replace(col("item_value_odd"), r'[\["\]]', ''))

# Left join even with odd on customer_id
df_joined = df_even.join(df_odd, on="customer_id", how="left")

display(df_joined)

In [0]:
df_joined.createOrReplaceTempView("df_joined")

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_silver.tbs_customer_item_values (
  customer_id STRING,
  item_name STRING NOT NULL,
  lat DOUBLE,
  lon DOUBLE,
  item_value STRING,
  PRIMARY KEY (item_name)
);

INSERT INTO end_to_end_retail.db_silver.tbs_customer_item_values
SELECT customer_id, item_value_even as item_name, lat, lon, item_value_odd as item_value
FROM df_joined
WHERE item_value_even IS NOT NULL;

In [0]:
%sql
SELECT 
  CONCAT(siv.customer_id, '_', siv.item_name, '_', siv.item_value) AS customer_item_concat,
  siv.item_name, 
  siv.customer_id,
  siv.item_value,
  CASE 
    WHEN siv.item_value LIKE '%96%' THEN 'SALES' 
    ELSE 'MAINTENANCE' 
  END AS type,
  cac.ship_to_address,
  cac.state,
  cac.city,
  cac.lon,
  cac.lat,
  ROW_NUMBER() OVER (PARTITION BY siv.customer_id ORDER BY CONCAT(siv.customer_id, '_', siv.item_name, '_', siv.item_value)) AS row_num
FROM end_to_end_retail.db_silver.tbs_customer_item_values siv
LEFT JOIN end_to_end_retail.db_gold.tbg_customer_address_clustering cac
  ON siv.customer_id = cac.customer_id
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_customer_item_values AS
SELECT DISTINCT
  CONCAT(siv.customer_id, '_', siv.item_name, '_', siv.item_value) AS customer_item_concat,
  siv.item_name, 
  siv.customer_id,
  siv.item_value,
  CASE 
    WHEN siv.item_value LIKE '%96%' THEN 'SALES' 
    ELSE 'MAINTENANCE' 
  END AS type,
  cac.ship_to_address,
  cac.state,
  cac.city,
  cac.lon,
  cac.lat
FROM end_to_end_retail.db_silver.tbs_customer_item_values siv
LEFT JOIN end_to_end_retail.db_gold.tbg_customer_address_clustering cac
  ON siv.customer_id = cac.customer_id

In [0]:
%sql
SELECT DISTINCT
  CONCAT(siv.customer_id, '_', siv.item_name, '_', siv.item_value) AS customer_item_concat,
  siv.item_name, 
  siv.customer_id,
  siv.item_value,
  CASE 
    WHEN TRY_CAST(siv.item_value AS DOUBLE) > 50 THEN 'MAINTENANCE'
    WHEN TRY_CAST(siv.item_value AS DOUBLE) <= 50 THEN 'SALES'
    ELSE 'MAINTENANCE'
  END AS type,
  cac.ship_to_address,
  cac.state,
  cac.city,
  cac.lon,
  cac.lat
FROM end_to_end_retail.db_silver.tbs_customer_item_values siv
LEFT JOIN end_to_end_retail.db_gold.tbg_customer_address_clustering cac
limit 5

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_customer_item_values_zipcode AS
SELECT 
  customer_item_concat as order_id,
  ship_to_address as street_address,
  SPLIT(ship_to_address, ',')[1] AS zip_code,
  lat as latitude,
  lon as longitude,
  type,
  item_value as value,
  current_timestamp() as last_processed_time
  -- customer_id,
  -- state,
  -- city,
FROM vw_customer_item_values

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_gold.tbg_customer_item_clustering
AS
SELECT 
  order_id,
  street_address,
  TRY_CAST(SUBSTRING(zip_code, 1, 6) AS INT) AS zipcode_int,
  zip_code,
  latitude,
  longitude,
  type,
  value,
  last_processed_time
FROM vw_customer_item_values_zipcode

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_gold.tbg_customer_item_clustering limit 10